In [1]:
#split columns

import pandas as pd
import os
import re
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()


silver_dir = os.getenv("SILVER")

cdate = datetime.now().strftime("%Y%m%d")
# 1. Path Configuration
excel_path = os.path.join(silver_dir, f"{cdate}.xlsx")


if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df.columns = df.columns.str.strip()

# Target column name variable (adjust if it's lowercase 'd' in your file)
target_col = 'Proposal Details' 

if target_col not in df.columns:
    raise KeyError(f"Could not find '{target_col}' column. Available columns are: {list(df.columns)}")

# 2. Define the Parsing Function
def extract_portal_details(text):
    # Fallback for empty/NaN cells
    if pd.isna(text):
        return pd.Series([None] * 5)
    
    text_str = str(text)
    
    # Using regex lookarounds to capture data between the known field labels
    clearance = re.search(r'Clearance Type:\s*(.*?)(?=\s*S/W No\.:|$)', text_str, re.IGNORECASE)
    sw_no     = re.search(r'S/W No\.\s*:\s*(.*?)(?=\s*Category:|$)', text_str, re.IGNORECASE)
    category  = re.search(r'Category\s*:\s*(.*?)(?=\s*Sector:|$)', text_str, re.IGNORECASE)
    sector    = re.search(r'Sector\s*:\s*(.*?)(?=\s*Date of Submission:|$)', text_str, re.IGNORECASE)
    date_sub  = re.search(r'Date of Submission\s*:\s*(.*?)$', text_str, re.IGNORECASE)
    
    # Extract match if found, strip trailing spaces, otherwise return None
    return pd.Series([
        clearance.group(1).strip() if clearance else None,
        sw_no.group(1).strip() if sw_no else None,
        category.group(1).strip() if category else None,
        sector.group(1).strip() if sector else None,
        date_sub.group(1).strip() if date_sub else None
    ])

# 3. Apply parsing to generate 5 new columns
print("Extracting data points from Proposal Details...")

new_cols = ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']
df[new_cols] = df[target_col].apply(extract_portal_details)

# 4. Save back to Excel
df.to_excel(excel_path, index=False)
print(f"Success! Extracted fields saved into columns: {new_cols}")

Extracting data points from Proposal Details...
Success! Extracted fields saved into columns: ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']


In [2]:
# Filling Sector and Year

from datetime import datetime
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

silver_dir = os.getenv("SILVER")
cdate = datetime.now().strftime("%Y%m%d")

# Define your file paths
input_path = os.path.join(silver_dir, f"{cdate}.xlsx")
output_path = os.path.join(silver_dir, f"{cdate}.xlsx")

# 1. Load the Excel file
df = pd.read_excel(input_path)

# Optional: Strip leading/trailing whitespaces to ensure exact matching
df["Activity Description"] = df["Activity Description"].astype(str).str.strip()

# 2. Build the lookup mapping from non-null Sector values
sector_lookup = (
    df.dropna(subset=["Sector"])
    .drop_duplicates(subset=["Activity Description"])
    .set_index("Activity Description")["Sector"]
    .to_dict()
)

# 3. Fill in missing Sector values using the Activity Description lookup
df["Sector"] = df["Sector"].fillna(df["Activity Description"].map(sector_lookup))

# 4. Extract Year from 'Date of Submission' column and add/update the 'Year' column
submission_col = None
for col in df.columns:
    if "date" in str(col).lower() and "submission" in str(col).lower():
        submission_col = col
        break

if submission_col:
    # Convert to datetime, extract the year, and handle invalid/missing dates gracefully
    extracted_years = pd.to_datetime(df[submission_col], errors="coerce").dt.year

    if "Year" in df.columns:
        df["Year"] = extracted_years.fillna(df["Year"])
        print("🔄 Updated existing 'Year' column from 'Date of Submission'.")
    else:
        # Find index of 'Date of Submission' column to place 'Year' right after it, otherwise append
        col_idx = df.columns.get_loc(submission_col) + 1
        df.insert(col_idx, "Year", extracted_years)
        print("➕ Inserted new 'Year' column right after 'Date of Submission'.")
else:
    print(
        "⚠️ Could not locate a 'Date of Submission' column to extract the Year."
    )

# 5. Save to the Excel file
df.to_excel(output_path, index=False)

print(f"Processing complete! Saved updated file to:\n{output_path}")

➕ Inserted new 'Year' column right after 'Date of Submission'.
Processing complete! Saved updated file to:
F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\New Pulls\20260809.xlsx


In [2]:
# Extract Form # and update status column
from datetime import datetime
import os
import re
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

cdate = datetime.now().strftime("%Y%m%d")
silver_dir = os.getenv("SILVER")
#Define path to target Excel file
file_path = os.path.join(silver_dir, f"{cdate}.xlsx")
#file_path = os.getenv("MDB")
if not os.path.exists(file_path):
    print(f"❌ File not found at path: {file_path}")
else:
    print(f"📂 Loading file: {file_path}")
    df = pd.read_excel(file_path)

    # 1. Strip non-printable ASCII control characters to avoid IllegalCharacterError
    illegal_xml_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")
    df = df.map(
        lambda x: illegal_xml_chars_re.sub("", x) if isinstance(x, str) else x
    )

    # 2. Function to extract Form No from the end of string
    def extract_form_number(text):
        if pd.isna(text):
            return None
        match = re.search(
            r"Form\s*[-–]?\s*([A-Za-z0-9]+)\s*$", str(text), re.IGNORECASE
        )
        return f"Form-{match.group(1)}" if match else None

    # Identify the clearance column name dynamically (handles variations)
    clearance_col = None
    for col in df.columns:
        if "clearance" in str(col).lower():
            clearance_col = col
            break

    # Identify the Proposal Status column dynamically (handles variations)
    status_col = None
    for col in df.columns:
        if "proposal" in str(col).lower() and "status" in str(col).lower():
            status_col = col
            break

    if clearance_col:
        print(f"🔍 Found target clearance column: '{clearance_col}'")

        form_series = df[clearance_col].apply(extract_form_number)

        # Check if "Form_No" already exists to prevent duplication errors on re-runs
        if "Form_No" in df.columns:
            df["Form_No"] = form_series
            print("🔄 Updated existing 'Form_No' column.")
        else:
            col_idx = df.columns.get_loc(clearance_col) + 1
            df.insert(col_idx, "Form_No", form_series)
            print("➕ Inserted new 'Form_No' column.")
    else:
        print("❌ Could not locate a 'Clearance Type' column in the Excel file.")

    if status_col:
        print(f"🔍 Found target status column: '{status_col}'")

        # Replace any value in Proposal Status starting with 'delisted' (case-insensitive) with 'Not Listed'
        df[status_col] = df[status_col].apply(
            lambda x: "Delisted"
            if isinstance(x, str)
            and re.match(r"^\s*delisted", x, re.IGNORECASE)
            else x
        )
    else:
        print(
            "❌ Could not locate a 'Proposal Status' column in the Excel file."
        )

    # 3. Save updates back to the same Excel file if any relevant column was processed
    if clearance_col or status_col:
        df.to_excel(file_path, index=False)
        print(f"\n✅ File updated successfully at:\n   {file_path}")
        print("\n--- Preview of updated columns ---")
        preview_cols = []
        if clearance_col:
            preview_cols.extend([clearance_col, "Form_No"])
        if status_col:
            preview_cols.append(status_col)
        print(df[preview_cols].head(10))

📂 Loading file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx
🔍 Found target clearance column: 'Clearance Type'
🔄 Updated existing 'Form_No' column.
🔍 Found target status column: 'Proposal Status'

✅ File updated successfully at:
   F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx

--- Preview of updated columns ---
                                      Clearance Type Form_No  \
0  Application for EC (Category A, B1, and B2 Vio...  Form-1   
1  Application for ToR (Category A, B1, and B2 Vi...  Form-1   
2  Application for EC (Category A, B1, and B2 Vio...  Form-1   
3  Application for EC (Category A, B1, and B2 Vio...  Form-1   
4  Application for EC (Category A, B1, and B2 Vio...  Form-1   
5  Application for ToR (Category A, B1, and B2 Vi...  Form-1   
6  Application for ToR (Category A, B1, and B2 Vi...  Form-1   
7  Application for EC for Mining of Minor Mineral...  Form-2   
8  Application for ToR (Category A, B1, and B2 

In [4]:
#Merge with masterDB

from datetime import datetime
import os
import pandas as pd

from dotenv import load_dotenv

load_dotenv()
# 1. Setup paths and dates from environment variables with fallback/validation
cdate = datetime.now().strftime("%Y%m%d")
silver_dir = os.getenv("SILVER")
master_db_path = os.getenv("MDB")
log_dir = os.getenv("SILVERLOG")

# Validate environment variables to prevent TypeError
if not silver_dir or not master_db_path or not log_dir:
  raise EnvironmentError(
      "One or more required environment variables (BRONZE, MDB, BRONZELOG) are not set."
  )

# Target daily pulled file path
file_path = os.path.join(silver_dir, f"{cdate}.xlsx")

# Ensure log directory exists
os.makedirs(log_dir, exist_ok=True)
log_file_path = os.path.join(log_dir, "processing_log.xlsx")

# 2. Read Master Database and Daily Pulled File
df_master = pd.read_excel(master_db_path)
df_daily = pd.read_excel(file_path)

# Ensure 'Comment' column exists in master DB; if not, initialize existing rows as 'initial records'
if "Comment" not in df_master.columns:
  df_master["Comment"] = "initial records"
else:
  df_master["Comment"] = df_master["Comment"].fillna("initial records")

# 3. Match columns: retain only the columns present in the Master DB (excluding 'Comment' for matching daily data)
master_columns = [col for col in df_master.columns if col != "Comment"]

# Filter daily dataframe to only include columns that exist in the master database
df_daily_filtered = df_daily[[col for col in master_columns if col in df_daily.columns]]

# 4. Identify new proposals by checking against the master database
existing_proposals = set(df_master["Proposal No."])
df_new = df_daily_filtered[~df_daily_filtered["Proposal No."].isin(existing_proposals)].copy()

num_new_records = len(df_new)

# 5. Add comment for new entries and append to master DB
if num_new_records > 0:
  df_new["Comment"] = cdate

  # Ensure columns align perfectly before concatenating
  df_updated_master = pd.concat([df_master, df_new], ignore_index=True)
else:
  df_updated_master = df_master

# Save updated master database back to excel
df_updated_master.to_excel(master_db_path, index=False)

# 6. Create or update the log entry in BRONZELOG
log_entry = pd.DataFrame(
    [{"Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "New Records Added": num_new_records}]
)

if os.path.exists(log_file_path):
  df_log = pd.read_excel(log_file_path)
  df_log = pd.concat([df_log, log_entry], ignore_index=True)
else:
  df_log = log_entry

df_log.to_excel(log_file_path, index=False)

print(f"Process completed successfully. {num_new_records} new records added.")

Process completed successfully. 22 new records added.
